In [14]:
import pandas as pd
import numpy as np
import os
import json
import re
import sys
sys.path.append('..')
from src.utils import calculate_metrics
from typing import List, Dict, Any

In [91]:
from ast import literal_eval

## Get incorrect results from eval

In [15]:
inference_path = '../outputs/evals3.1/eap/circuit-indo_finetune-indo/seed_123/aos_sequence_variants/2025-10-08 01:11:42.891178_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/inference_results.json'
with open(inference_path, 'r') as f:
	inference_results = json.load(f)

In [16]:
def parse_aos_triplet(triplet_str):
    """
    Parse AOS triplet string to extract Aspect, Opinion, and Sentiment.
    Works with any ordering of [A], [O], [S] tags and any length of text between them.
    
    Args:
        triplet_str: String like "[A] service [O] very good [S] positive" or
                    "[O] very good [A] service [S] positive" or any other ordering
    
    Returns:
        tuple: (aspect, opinion, sentiment) or None if parsing fails
    """
    # Remove extra whitespaces and strip
    triplet_str = ' '.join(triplet_str.split()).strip()
    
    # Initialize variables
    aspect = None
    opinion = None
    sentiment = None
    
    # Find all tags and their positions
    tags = ['A', 'O', 'S']
    tag_positions = {}
    
    for tag in tags:
        pattern = rf'\[{tag}\]'
        match = re.search(pattern, triplet_str)
        if match:
            tag_positions[tag] = match.start()
        else:
            # Missing tag, cannot parse
            return None
    
    # Sort tags by their positions in the string
    sorted_tags = sorted(tag_positions.items(), key=lambda x: x[1])
    
    # Extract content between tags
    for i, (tag, pos) in enumerate(sorted_tags):
        # Find start position (after the tag)
        tag_end = pos + len(f'[{tag}]')
        
        # Find end position (start of next tag or end of string)
        if i < len(sorted_tags) - 1:
            next_tag_pos = sorted_tags[i + 1][1]
            content = triplet_str[tag_end:next_tag_pos]
        else:
            content = triplet_str[tag_end:]
        
        # Clean up content
        content = content.strip()
        
        # Map to correct variable based on tag
        if tag == 'A':
            aspect = content
        elif tag == 'O':
            opinion = content
        elif tag == 'S':
            sentiment = content
    
    # Return tuple if all three components found
    if aspect is not None and opinion is not None and sentiment is not None:
        return (aspect, opinion, sentiment)
    else:
        return None

def parse_multiple_triplets(triplet_str, separator='[SSEP]'):
    """
    Parse multiple AOS triplets separated by a delimiter.
    
    Args:
        triplet_str: String with multiple triplets like "[A] service [O] good [S] positive [SSEP] [A] place [O] nice [S] positive"
        separator: Separator between triplets (default: '[SSEP]')
    
    Returns:
        list: List of (aspect, opinion, sentiment) tuples
    """
    # Split by separator and parse each triplet
    triplets_str = triplet_str.split(separator)
    triplets = []
    
    for t in triplets_str:
        t = t.strip()
        if t == '':
            return [] 
            
        parsed = parse_aos_triplet(t)
        if parsed:
            triplets.append(parsed)
        else:
            print(f"Warning: Could not parse triplet: '{t}'")
    
    return triplets

In [17]:
inference_results[0]

{'sentence_id': 3500,
 'task_elements': 'aos',
 'element_order': 'aos',
 'input': 'pelayanan nya sangat ramah . [A] [O] [S]',
 'target': '[A] pelayanan nya [O] sangat ramah [S] positive',
 'prediction': '[A] pelayanan nya [O] sangat ramah [S] positive',
 'target_list': ['[A] pelayanan nya [O] sangat ramah [S] positive'],
 'prediction_list': ['[A] pelayanan nya [O] sangat ramah [S] positive']}

In [18]:
prompt = 'pelayanan nya sangat ramah . [A] [O] [S]'
label = '[A] pelayanan nya [O] sangat ramah [S] positive [SSEP] [A] harga nya [O] terjangkau [S] positive'
pred = '[A] pelayanan nya [O] sangat ramah [S] positive [SSEP] [A] harga nya [O] terjangkau'
match = re.search(r"(\[[A-Z]\](\s)*)+$", prompt)
temp_task = match.group().strip()
task = re.sub(r"[\[\]\s]", "", temp_task).lower()
target_split = label.split(" [SSEP] ")
pred_clean = pred.split(temp_task+" ")[-1]
pred_split = pred_clean.split(" [SSEP] ")

In [19]:
def calculate_metrics(predictions: List[List[Dict[str, str]]], targets: List[List[Dict[str, str]]], task='') -> Dict[str, float]:
    """
    Calculate precision, recall, and F1 score for the given predictions and targets for ABSA.

    Args:
        predictions (List[List[Dict[str, str]]]): List of predicted triplets.
        targets (List[List[Dict[str, str]]]): List of target triplets.
        task (str): The task name for which metrics are calculated.
    
    Returns:
        Dict[str, float]: A dictionary containing precision, recall, and F1 score.
    """
    true_positive = 0
    false_positive = 0
    false_negative = 0
    # print(f'predictions: {predictions} targets: {targets}')
    for prediction,target in zip(predictions,targets):
        for target_tuple in target:
            if target_tuple in prediction:
                true_positive += 1
            else:
                false_negative += 1
        false_positive += sum(1 for pred in prediction if pred not in target)
    # print(f"TP: {true_positive}, FP: {false_positive}, FN: {false_negative}")
    precision = true_positive/(true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
    recall = true_positive/(true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
    f1 = (2 * recall * precision)/(recall + precision) if (recall + precision) > 0 else 0
    return {
        f"precision_{task}" : precision,
        f"recall_{task}" : recall,
        f"f1_{task}" : f1
    }

In [20]:
# if task not in per_task.keys():
# 	per_task[task] = {"predictions": [], "targets":[]}
# per_task[task]["predictions"].append(pred_split)
# per_task[task]["targets"].append(target_split)
# result_metrics = {}
# for task, v in per_task.items():
# 	predictions = v["predictions"]
# 	targets = v["targets"]
# 	result_metrics.update(
# 		calculate_metrics(predictions, targets, task)
# )
# scaled_result_metrics = {key: value * 100 for key, value in result_metrics.items()}

In [21]:
inference_results[0]

{'sentence_id': 3500,
 'task_elements': 'aos',
 'element_order': 'aos',
 'input': 'pelayanan nya sangat ramah . [A] [O] [S]',
 'target': '[A] pelayanan nya [O] sangat ramah [S] positive',
 'prediction': '[A] pelayanan nya [O] sangat ramah [S] positive',
 'target_list': ['[A] pelayanan nya [O] sangat ramah [S] positive'],
 'prediction_list': ['[A] pelayanan nya [O] sangat ramah [S] positive']}

In [22]:
false_inference_results = []
temp_results_per_sentence = []
prediction_status_per_order = {}
wrong_prediction = False
for index, result in enumerate(inference_results):
	temp_results_per_sentence.append(result)
	task = result['element_order']
	predictions = [result['prediction_list']]
	targets = [result['target_list']]
	
	# f1_score check
	temp_results_per_sentence[index % 5]['f1_score'] = calculate_metrics(predictions, targets, task)[f'f1_{task}']
	if temp_results_per_sentence[index % 5]['f1_score'] < 1.0:
		wrong_prediction = True

	# Round the f1_score to 4 decimal places
	temp_results_per_sentence[index % 5]['f1_score'] = round(temp_results_per_sentence[index % 5]['f1_score'], 4)

	# # Exact match check
	# if set(result['target_list']) != set(result['prediction_list']):
	# 	wrong_prediction = True
	# 	temp_results_per_sentence[index % 5]['correctness'] = False
	# else:
	# 	temp_results_per_sentence[index % 5]['correctness'] = True

	if index % 5 == 4:  # Process every 5th result
		if wrong_prediction:
			false_inference_results.extend(temp_results_per_sentence)
		temp_results_per_sentence = []
		wrong_prediction = False

In [23]:
len(false_inference_results)

3645

In [24]:
data = {
	'sentence_id': [],
	'input': [],
	'target': [],
	'correctness': [],
}
correctness_per_sentence = {}
for index, result in enumerate(false_inference_results):
	order_key = f'prediction_{result["element_order"]}'
	if order_key not in data.keys():
		data[order_key] = []
	correctness_per_sentence[f'f1_{result["element_order"]}'] = result['f1_score']
	parsed_triplets = parse_multiple_triplets(result['prediction'])
	parsed_targets = parse_multiple_triplets(result['target'])
	if index % 5 == 4: # Process every 5th result
		data['sentence_id'].append(result['sentence_id'])
		data['input'].append(result['input'].replace('[A]', '').replace('[O]', '').replace('[S]', '').strip())
		data['target'].append('\n'.join(str(t) for t in parsed_targets))
		data['correctness'].append(json.dumps(correctness_per_sentence, indent=2, ensure_ascii=False))
		correctness_per_sentence = {}
	data[order_key].append('\n'.join(str(t) for t in parsed_triplets))

In [ ]:
df_inference = pd.DataFrame(data)
df_inference.to_csv('formatted_false_inference_results_splitopinion_typocorrected_seed123_topk2000.csv', index=False)

## Compare between two results

In [124]:
df_topk = pd.read_csv('../test/formatted_false_inference_results_splitopinion_typocorrected_seed123_topk5000.csv')
df_full = pd.read_csv('../test/formatted_false_inference_results_splitopinion_typocorrected_seed123_fullsft.csv')

In [125]:
df_topk

,sentence_id,input,target,correctness,prediction_aos,prediction_aso,prediction_sao,prediction_oas,prediction_osa
0,3502,"tulisannya twin bed , tetapi yang ada kamarnya...","('kamarnya', 'beda', 'negative')","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('kamarnya', 'bersih', 'positive')\n('kamarnya...","('kamarnya', 'bersih', 'negative')\n('kamarnya...","('null', 'twin bed , tetapi yang ada kamarnya ...","('null', 'twin bed , tetapi yang ada kamarnya ...","('kamarnya', 'twin bed , tetapi yang ada kamar..."
1,3503,"over all baik , hanya sja akan lebih memuaskan...","('over all', 'baik', 'positive')\n('air hot wa...","{\n ""f1_aos"": 0.4,\n ""f1_aso"": 0.4,\n ""f1_s...","('over all', 'baik', 'positive')\n('air shower...","('over all', 'baik', 'positive')\n('air shower...","('over all', 'baik', 'positive')\n('air hot wa...","('over all', 'baik', 'positive')\n('air hot wa...","('over all', 'baik', 'positive')\n('air hot wa..."
2,3504,fasilatas sesuia .,"('fasilitas', 'sesuai', 'positive')","{\n ""f1_aos"": 1.0,\n ""f1_aso"": 1.0,\n ""f1_s...","('fasilitas', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')","('null', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')"
3,3505,dekat akses transportasi .,"('null', 'dekat akses transportasi', 'positive')","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('null', 'dekat akses mobilasi', 'positive')","('null', 'dekat akses', 'positive')","('null', 'dekat akses', 'positive')","('null', 'dekat akses tolong', 'positive')","('null', 'dekat akses tolong', 'positive')"
4,3506,kamar bagus seasui bajet .,"('kamar', 'bagus', 'positive')\n('kamar', 'ses...","{\n ""f1_aos"": 0.4,\n ""f1_aso"": 0.4,\n ""f1_s...","('kamar', 'bagus', 'positive')\n('kamar', 'ses...","('kamar', 'bagus', 'positive')\n('kamar', 'ses...","('kamar', 'bagus', 'positive')\n('kamar', 'ses...","('kamar', 'bagus', 'positive')\n('kamar', 'ses...","('kamar', 'bagus', 'positive')\n('kamar', 'ses..."
...,...,...,...,...,...,...,...,...,...
732,4494,harga terjangkau dengan fasilitas yang sangat ...,"('harga', 'terjangkau', 'positive')\n('fasilit...","{\n ""f1_aos"": 1.0,\n ""f1_aso"": 1.0,\n ""f1_s...","('harga', 'terjangkau', 'positive')\n('fasilit...","('harga', 'terjangkau', 'positive')\n('fasilit...","('harga', 'terjangkau', 'positive')\n('fasilit...","('harga', 'terjangkau dengan fasilitas yang sa...","('harga', 'terjangkau dengan fasilitas yang sa..."
733,4496,buat lakilaki dan perempuan yang belum menikah...,"('null', 'buat laki laki dan perempuan yang be...","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('kamar', 'tidak dapat', 'negative')\n('perala...","('kamar', 'kalau mau menginap perlu di tambahk...","('kamar', 'kalau mau menginap perlu di tambahk...","('kamar', 'tidak dapat', 'negative')\n('peremp...","('kamar', 'tidak dapat', 'negative')\n('perala..."
734,4497,"kamar sangat nyaman dan bersih , sungguh menye...","('kamar', 'sangat nyaman', 'positive')\n('kama...","{\n ""f1_aos"": 0.75,\n ""f1_aso"": 0.75,\n ""f1...","('kamar', 'sangat nyaman', 'positive')\n('kama...","('kamar', 'sangat nyaman', 'positive')\n('kama...","('kamar', 'sangat nyaman', 'positive')\n('kama...","('kamar', 'sangat nyaman', 'positive')\n('kama...","('kamar', 'sangat nyaman', 'positive')\n('kama..."
735,4498,"kamarnya luas , kasurnya empuk , kamar mandiny...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","{\n ""f1_aos"": 0.625,\n ""f1_aso"": 0.75,\n ""f...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","('kamarnya', 'luas', 'positive')\n('kasurnya',..."


In [126]:
df_full    

,sentence_id,input,target,correctness,prediction_aos,prediction_aso,prediction_sao,prediction_oas,prediction_osa
0,3501,sayang wifi tidak bagus harus keluar kamar .,"('wifi', 'tidak bagus harus keluar kamar', 'ne...","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ..."
1,3502,"tulisannya twin bed , tetapi yang ada kamarnya...","('kamarnya', 'beda', 'negative')","{\n ""f1_aos"": 0.6667,\n ""f1_aso"": 0.6667,\n ...","('null', 'twin bed', 'negative')\n('kamarnya',...","('bed', 'suite bed', 'negative')\n('kamarnya',...","('null', 'beda', 'positive')\n('kamarnya', 'ad...","('null', 'beda', 'negative')","('kamarnya', 'beda', 'negative')\n('tiletnya',..."
2,3503,"over all baik , hanya sja akan lebih memuaskan...","('over all', 'baik', 'positive')\n('air hot wa...","{\n ""f1_aos"": 0.3333,\n ""f1_aso"": 0.3333,\n ...","('over all', 'baik', 'positive')\n('air', 'sis...","('over all', 'baik', 'positive')\n('air', 'han...","('over all', 'baik', 'positive')\n('air', 'han...","('over all', 'baik', 'positive')\n('air', 'oke...","('over all', 'baik', 'positive')\n('sana', 'ok..."
3,3505,dekat akses transportasi .,"('null', 'dekat akses transportasi', 'positive')","{\n ""f1_aos"": 1.0,\n ""f1_aso"": 0,\n ""f1_sao...","('null', 'dekat akses transportasi', 'positive')","('null', 'dekat akses mobil aksesibilitasi', '...","('null', 'dekat akses mobil akses', 'positive')","('null', 'dekat akses transportasi', 'positive')","('null', 'dekat akses transportasi', 'positive')"
4,3506,kamar bagus seasui bajet .,"('kamar', 'bagus', 'positive')\n('kamar', 'ses...","{\n ""f1_aos"": 0.5,\n ""f1_aso"": 0.5,\n ""f1_s...","('kamar', 'bagus', 'positive')\n('kamar', 'set...","('kamar', 'bagus', 'positive')\n('kamar', 'bes...","('kamar', 'bagus', 'positive')\n('kamar', 'sed...","('kamar', 'bagus', 'positive')\n('kamar', 'set...","('kamar', 'bagus', 'positive')\n('kamar', 'set..."
...,...,...,...,...,...,...,...,...,...
724,4495,"lumayan , harga murah banged .","('harga', 'murah banget', 'positive')\n('null'...","{\n ""f1_aos"": 0.5,\n ""f1_aso"": 1.0,\n ""f1_s...","('harga', 'murah', 'positive')\n('null', 'luma...","('harga', 'murah banget', 'positive')\n('null'...","('harga', 'murah banget', 'positive')\n('null'...","('harga', 'murah', 'positive')\n('null', 'luma...","('harga', 'murah banget', 'positive')\n('null'..."
725,4496,buat lakilaki dan perempuan yang belum menikah...,"('null', 'buat laki laki dan perempuan yang be...","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('wifi', 'tidak dapat', 'negative')\n('breakfa...","('wifi', 'tidak dapat', 'negative')\n('breakfa...","('wifi', 'tidak dapat', 'negative')\n('breakfa...","('wifi', 'tidak dapat', 'negative')\n('breakfa...","('wifi', 'tidak dapat', 'negative')\n('breakfa..."
726,4497,"kamar sangat nyaman dan bersih , sungguh menye...","('kamar', 'sangat nyaman', 'positive')\n('kama...","{\n ""f1_aos"": 0.75,\n ""f1_aso"": 1.0,\n ""f1_...","('kamar', 'sangat nyaman', 'positive')\n('kama...","('kamar', 'sangat nyaman', 'positive')\n('kama...","('kamar', 'sangat nyaman', 'positive')\n('kama...","('kamar', 'sangat nyaman', 'positive')\n('kama...","('kamar', 'sangat nyaman', 'positive')\n('kama..."
727,4498,"kamarnya luas , kasurnya empuk , kamar mandiny...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","{\n ""f1_aos"": 0.75,\n ""f1_aso"": 0.625,\n ""f...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","('kamarnya', 'luas', 'positive')\n('kasurnya',..."


In [127]:
# Merge two dataframes on sentence_id

df_merged = pd.merge(df_topk, df_full, how='outer', on='sentence_id', suffixes=('_topk', '_full'))
df_merged

,sentence_id,input_topk,target_topk,correctness_topk,prediction_aos_topk,prediction_aso_topk,prediction_sao_topk,prediction_oas_topk,prediction_osa_topk,input_full,target_full,correctness_full,prediction_aos_full,prediction_aso_full,prediction_sao_full,prediction_oas_full,prediction_osa_full
0,3501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sayang wifi tidak bagus harus keluar kamar .,"('wifi', 'tidak bagus harus keluar kamar', 'ne...","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ..."
1,3502,"tulisannya twin bed , tetapi yang ada kamarnya...","('kamarnya', 'beda', 'negative')","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('kamarnya', 'bersih', 'positive')\n('kamarnya...","('kamarnya', 'bersih', 'negative')\n('kamarnya...","('null', 'twin bed , tetapi yang ada kamarnya ...","('null', 'twin bed , tetapi yang ada kamarnya ...","('kamarnya', 'twin bed , tetapi yang ada kamar...","tulisannya twin bed , tetapi yang ada kamarnya...","('kamarnya', 'beda', 'negative')","{\n ""f1_aos"": 0.6667,\n ""f1_aso"": 0.6667,\n ...","('null', 'twin bed', 'negative')\n('kamarnya',...","('bed', 'suite bed', 'negative')\n('kamarnya',...","('null', 'beda', 'positive')\n('kamarnya', 'ad...","('null', 'beda', 'negative')","('kamarnya', 'beda', 'negative')\n('tiletnya',..."
2,3503,"over all baik , hanya sja akan lebih memuaskan...","('over all', 'baik', 'positive')\n('air hot wa...","{\n ""f1_aos"": 0.4,\n ""f1_aso"": 0.4,\n ""f1_s...","('over all', 'baik', 'positive')\n('air shower...","('over all', 'baik', 'positive')\n('air shower...","('over all', 'baik', 'positive')\n('air hot wa...","('over all', 'baik', 'positive')\n('air hot wa...","('over all', 'baik', 'positive')\n('air hot wa...","over all baik , hanya sja akan lebih memuaskan...","('over all', 'baik', 'positive')\n('air hot wa...","{\n ""f1_aos"": 0.3333,\n ""f1_aso"": 0.3333,\n ...","('over all', 'baik', 'positive')\n('air', 'sis...","('over all', 'baik', 'positive')\n('air', 'han...","('over all', 'baik', 'positive')\n('air', 'han...","('over all', 'baik', 'positive')\n('air', 'oke...","('over all', 'baik', 'positive')\n('sana', 'ok..."
3,3504,fasilatas sesuia .,"('fasilitas', 'sesuai', 'positive')","{\n ""f1_aos"": 1.0,\n ""f1_aso"": 1.0,\n ""f1_s...","('fasilitas', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')","('null', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3505,dekat akses transportasi .,"('null', 'dekat akses transportasi', 'positive')","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('null', 'dekat akses mobilasi', 'positive')","('null', 'dekat akses', 'positive')","('null', 'dekat akses', 'positive')","('null', 'dekat akses tolong', 'positive')","('null', 'dekat akses tolong', 'positive')",dekat akses transportasi .,"('null', 'dekat akses transportasi', 'positive')","{\n ""f1_aos"": 1.0,\n ""f1_aso"": 0,\n ""f1_sao...","('null', 'dekat akses transportasi', 'positive')","('null', 'dekat akses mobil aksesibilitasi', '...","('null', 'dekat akses mobil akses', 'positive')","('null', 'dekat akses transportasi', 'positive')","('null', 'dekat akses transportasi', 'positive')"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
774,4495,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"lumayan , harga murah banged .","('harga', 'murah banget', 'positive')\n('null'...","{\n ""f1_aos"": 0.5,\n ""f1_aso"": 1.0,\n ""f1_s...","('harga', 'murah', 'positive')\n('null', 'luma...","('harga', 'murah banget', 'positive')\n('null'...","('harga', 'murah banget', 'positive')\n('null'...","('harga', 'murah', 'positive')\n('null', 'luma...","('harga', 'murah banget', 'positive')\n('null'..."
775,4496,buat lakilaki dan perempuan 

In [128]:
df_merged['correctness_full'] = df_merged['correctness_full'].fillna({
    'f1_aos': 1.0,
    'f1_aso': 1.0,
    'f1_sao': 1.0,
    'f1_oas': 1.0,
    'f1_osa': 1.0
})

df_merged['correctness_topk'] = df_merged['correctness_topk'].fillna({
    'f1_aos': 1.0,
    'f1_aso': 1.0,
    'f1_sao': 1.0,
    'f1_oas': 1.0,
    'f1_osa': 1.0
})

In [129]:
df_merged['input_topk'] = df_merged['input_topk'].fillna(df_merged['input_full'])
df_merged['target_topk'] = df_merged['target_topk'].fillna(df_merged['target_full'])
df_merged['input_full'] = df_merged['input_full'].fillna(df_merged['input_topk'])
df_merged['target_full'] = df_merged['target_full'].fillna(df_merged['target_topk'])
df_merged

,sentence_id,input_topk,target_topk,correctness_topk,prediction_aos_topk,prediction_aso_topk,prediction_sao_topk,prediction_oas_topk,prediction_osa_topk,input_full,target_full,correctness_full,prediction_aos_full,prediction_aso_full,prediction_sao_full,prediction_oas_full,prediction_osa_full
0,3501,sayang wifi tidak bagus harus keluar kamar .,"('wifi', 'tidak bagus harus keluar kamar', 'ne...",NaN,NaN,NaN,NaN,NaN,NaN,sayang wifi tidak bagus harus keluar kamar .,"('wifi', 'tidak bagus harus keluar kamar', 'ne...","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ..."
1,3502,"tulisannya twin bed , tetapi yang ada kamarnya...","('kamarnya', 'beda', 'negative')","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('kamarnya', 'bersih', 'positive')\n('kamarnya...","('kamarnya', 'bersih', 'negative')\n('kamarnya...","('null', 'twin bed , tetapi yang ada kamarnya ...","('null', 'twin bed , tetapi yang ada kamarnya ...","('kamarnya', 'twin bed , tetapi yang ada kamar...","tulisannya twin bed , tetapi yang ada kamarnya...","('kamarnya', 'beda', 'negative')","{\n ""f1_aos"": 0.6667,\n ""f1_aso"": 0.6667,\n ...","('null', 'twin bed', 'negative')\n('kamarnya',...","('bed', 'suite bed', 'negative')\n('kamarnya',...","('null', 'beda', 'positive')\n('kamarnya', 'ad...","('null', 'beda', 'negative')","('kamarnya', 'beda', 'negative')\n('tiletnya',..."
2,3503,"over all baik , hanya sja akan lebih memuaskan...","('over all', 'baik', 'positive')\n('air hot wa...","{\n ""f1_aos"": 0.4,\n ""f1_aso"": 0.4,\n ""f1_s...","('over all', 'baik', 'positive')\n('air shower...","('over all', 'baik', 'positive')\n('air shower...","('over all', 'baik', 'positive')\n('air hot wa...","('over all', 'baik', 'positive')\n('air hot wa...","('over all', 'baik', 'positive')\n('air hot wa...","over all baik , hanya sja akan lebih memuaskan...","('over all', 'baik', 'positive')\n('air hot wa...","{\n ""f1_aos"": 0.3333,\n ""f1_aso"": 0.3333,\n ...","('over all', 'baik', 'positive')\n('air', 'sis...","('over all', 'baik', 'positive')\n('air', 'han...","('over all', 'baik', 'positive')\n('air', 'han...","('over all', 'baik', 'positive')\n('air', 'oke...","('over all', 'baik', 'positive')\n('sana', 'ok..."
3,3504,fasilatas sesuia .,"('fasilitas', 'sesuai', 'positive')","{\n ""f1_aos"": 1.0,\n ""f1_aso"": 1.0,\n ""f1_s...","('fasilitas', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')","('null', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')",fasilatas sesuia .,"('fasilitas', 'sesuai', 'positive')",NaN,NaN,NaN,NaN,NaN,NaN
4,3505,dekat akses transportasi .,"('null', 'dekat akses transportasi', 'positive')","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('null', 'dekat akses mobilasi', 'positive')","('null', 'dekat akses', 'positive')","('null', 'dekat akses', 'positive')","('null', 'dekat akses tolong', 'positive')","('null', 'dekat akses tolong', 'positive')",dekat akses transportasi .,"('null', 'dekat akses transportasi', 'positive')","{\n ""f1_aos"": 1.0,\n ""f1_aso"": 0,\n ""f1_sao...","('null', 'dekat akses transportasi', 'positive')","('null', 'dekat akses mobil aksesibilitasi', '...","('null', 'dekat akses mobil akses', 'positive')","('null', 'dekat akses transportasi', 'positive')","('null', 'dekat akses transportasi', 'positive')"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
774,4495,"lumayan , harga murah banged .","('harga', 'murah banget', 'positive')\n('null'...",NaN,NaN,NaN,NaN,NaN,NaN,"lumayan , harga murah banged .","('harga', 'murah banget', 'positive')\n('null'...","{\n ""f1_aos"": 0.5,\n ""f1_aso"": 1.0,\n ""f1_s...","('harga', 'murah', 'positive')\n('null', 'luma...","('harga', 'murah banget', 'po

In [130]:
topk_win = []
for idx, row in df_merged.iterrows():
	if pd.isna(row['correctness_topk']):
		f1_topk = 1.0
	else:
		f1_topk = literal_eval(row['correctness_topk'])['f1_aos']
	if pd.isna(row['correctness_full']):
		f1_full = 1.0
	else:
		f1_full = literal_eval(row['correctness_full'])['f1_aos']
	
	if f1_topk > f1_full:
		topk_win.append('win')
	elif f1_topk == f1_full:
		topk_win.append('draw')
	else:
		topk_win.append('lose')


In [131]:
df_merged['topk_win'] = topk_win

In [132]:
df_merged

,sentence_id,input_topk,target_topk,correctness_topk,prediction_aos_topk,prediction_aso_topk,prediction_sao_topk,prediction_oas_topk,prediction_osa_topk,input_full,target_full,correctness_full,prediction_aos_full,prediction_aso_full,prediction_sao_full,prediction_oas_full,prediction_osa_full,topk_win
0,3501,sayang wifi tidak bagus harus keluar kamar .,"('wifi', 'tidak bagus harus keluar kamar', 'ne...",NaN,NaN,NaN,NaN,NaN,NaN,sayang wifi tidak bagus harus keluar kamar .,"('wifi', 'tidak bagus harus keluar kamar', 'ne...","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...",win
1,3502,"tulisannya twin bed , tetapi yang ada kamarnya...","('kamarnya', 'beda', 'negative')","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('kamarnya', 'bersih', 'positive')\n('kamarnya...","('kamarnya', 'bersih', 'negative')\n('kamarnya...","('null', 'twin bed , tetapi yang ada kamarnya ...","('null', 'twin bed , tetapi yang ada kamarnya ...","('kamarnya', 'twin bed , tetapi yang ada kamar...","tulisannya twin bed , tetapi yang ada kamarnya...","('kamarnya', 'beda', 'negative')","{\n ""f1_aos"": 0.6667,\n ""f1_aso"": 0.6667,\n ...","('null', 'twin bed', 'negative')\n('kamarnya',...","('bed', 'suite bed', 'negative')\n('kamarnya',...","('null', 'beda', 'positive')\n('kamarnya', 'ad...","('null', 'beda', 'negative')","('kamarnya', 'beda', 'negative')\n('tiletnya',...",lose
2,3503,"over all baik , hanya sja akan lebih memuaskan...","('over all', 'baik', 'positive')\n('air hot wa...","{\n ""f1_aos"": 0.4,\n ""f1_aso"": 0.4,\n ""f1_s...","('over all', 'baik', 'positive')\n('air shower...","('over all', 'baik', 'positive')\n('air shower...","('over all', 'baik', 'positive')\n('air hot wa...","('over all', 'baik', 'positive')\n('air hot wa...","('over all', 'baik', 'positive')\n('air hot wa...","over all baik , hanya sja akan lebih memuaskan...","('over all', 'baik', 'positive')\n('air hot wa...","{\n ""f1_aos"": 0.3333,\n ""f1_aso"": 0.3333,\n ...","('over all', 'baik', 'positive')\n('air', 'sis...","('over all', 'baik', 'positive')\n('air', 'han...","('over all', 'baik', 'positive')\n('air', 'han...","('over all', 'baik', 'positive')\n('air', 'oke...","('over all', 'baik', 'positive')\n('sana', 'ok...",win
3,3504,fasilatas sesuia .,"('fasilitas', 'sesuai', 'positive')","{\n ""f1_aos"": 1.0,\n ""f1_aso"": 1.0,\n ""f1_s...","('fasilitas', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')","('null', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')",fasilatas sesuia .,"('fasilitas', 'sesuai', 'positive')",NaN,NaN,NaN,NaN,NaN,NaN,draw
4,3505,dekat akses transportasi .,"('null', 'dekat akses transportasi', 'positive')","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('null', 'dekat akses mobilasi', 'positive')","('null', 'dekat akses', 'positive')","('null', 'dekat akses', 'positive')","('null', 'dekat akses tolong', 'positive')","('null', 'dekat akses tolong', 'positive')",dekat akses transportasi .,"('null', 'dekat akses transportasi', 'positive')","{\n ""f1_aos"": 1.0,\n ""f1_aso"": 0,\n ""f1_sao...","('null', 'dekat akses transportasi', 'positive')","('null', 'dekat akses mobil aksesibilitasi', '...","('null', 'dekat akses mobil akses', 'positive')","('null', 'dekat akses transportasi', 'positive')","('null', 'dekat akses transportasi', 'positive')",lose
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
774,4495,"lumayan , harga murah banged .","('harga', 'murah banget', 'positive')\n('null'...",NaN,NaN,NaN,NaN,NaN,NaN,"lumayan , harga murah banged .","('harga', 'murah banget', 'positive')\n('null'...","{\n ""f1_aos"": 0.5,\n ""f1_aso"": 1.0,\n ""f1_s...","('harga', 'murah', 'positive')\n('null', 'lum

In [133]:
df_merged['prediction_aos_topk'] = df_merged['prediction_aos_topk'].fillna(df_merged['target_topk'])
df_merged['prediction_aos_full'] = df_merged['prediction_aos_full'].fillna(df_merged['target_full'])

In [134]:
df_merged[['sentence_id', 'input_topk', 'target_topk', 'correctness_topk', 'correctness_full', 'prediction_aos_topk', 'prediction_aos_full', 'topk_win']]

,sentence_id,input_topk,target_topk,correctness_topk,correctness_full,prediction_aos_topk,prediction_aos_full,topk_win
0,3501,sayang wifi tidak bagus harus keluar kamar .,"('wifi', 'tidak bagus harus keluar kamar', 'ne...",NaN,"{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('wifi', 'tidak bagus harus keluar kamar', 'ne...","('wifi', 'tidak bagus', 'negative')\n('wifi', ...",win
1,3502,"tulisannya twin bed , tetapi yang ada kamarnya...","('kamarnya', 'beda', 'negative')","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","{\n ""f1_aos"": 0.6667,\n ""f1_aso"": 0.6667,\n ...","('kamarnya', 'bersih', 'positive')\n('kamarnya...","('null', 'twin bed', 'negative')\n('kamarnya',...",lose
2,3503,"over all baik , hanya sja akan lebih memuaskan...","('over all', 'baik', 'positive')\n('air hot wa...","{\n ""f1_aos"": 0.4,\n ""f1_aso"": 0.4,\n ""f1_s...","{\n ""f1_aos"": 0.3333,\n ""f1_aso"": 0.3333,\n ...","('over all', 'baik', 'positive')\n('air shower...","('over all', 'baik', 'positive')\n('air', 'sis...",win
3,3504,fasilatas sesuia .,"('fasilitas', 'sesuai', 'positive')","{\n ""f1_aos"": 1.0,\n ""f1_aso"": 1.0,\n ""f1_s...",NaN,"('fasilitas', 'sesuai', 'positive')","('fasilitas', 'sesuai', 'positive')",draw
4,3505,dekat akses transportasi .,"('null', 'dekat akses transportasi', 'positive')","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","{\n ""f1_aos"": 1.0,\n ""f1_aso"": 0,\n ""f1_sao...","('null', 'dekat akses mobilasi', 'positive')","('null', 'dekat akses transportasi', 'positive')",lose
...,...,...,...,...,...,...,...,...
774,4495,"lumayan , harga murah banged .","('harga', 'murah banget', 'positive')\n('null'...",NaN,"{\n ""f1_aos"": 0.5,\n ""f1_aso"": 1.0,\n ""f1_s...","('harga', 'murah banget', 'positive')\n('null'...","('harga', 'murah', 'positive')\n('null', 'luma...",win
775,4496,buat lakilaki dan perempuan yang belum menikah...,"('null', 'buat laki laki dan perempuan yang be...","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","{\n ""f1_aos"": 0,\n ""f1_aso"": 0,\n ""f1_sao"":...","('kamar', 'tidak dapat', 'negative')\n('perala...","('wifi', 'tidak dapat', 'negative')\n('breakfa...",draw
776,4497,"kamar sangat nyaman dan bersih , sungguh menye...","('kamar', 'sangat nyaman', 'positive')\n('kama...","{\n ""f1_aos"": 0.75,\n ""f1_aso"": 0.75,\n ""f1...","{\n ""f1_aos"": 0.75,\n ""f1_aso"": 1.0,\n ""f1_...","('kamar', 'sangat nyaman', 'positive')\n('kama...","('kamar', 'sangat nyaman', 'positive')\n('kama...",draw
777,4498,"kamarnya luas , kasurnya empuk , kamar mandiny...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","{\n ""f1_aos"": 0.625,\n ""f1_aso"": 0.75,\n ""f...","{\n ""f1_aos"": 0.75,\n ""f1_aso"": 0.625,\n ""f...","('kamarnya', 'luas', 'positive')\n('kasurnya',...","('kamarnya', 'luas', 'positive')\n('kasurnya',...",lose


In [135]:
df_merged[['sentence_id', 'input_topk', 'target_topk', 'correctness_topk', 'correctness_full', 'prediction_aos_topk', 'prediction_aos_full', 'topk_win']].to_csv('comparison_topk5000_fullsft_splitopinion_typocorrected_seed123.csv', index=False)